# Keep last version, and keep only the 100 initially coded (not the ones in the google sheet)

In [12]:
import pandas as pd

# Define the exclusion list - case numbers to filter out
exclude_cases = [
    '23CHLC18998', '23CHLC16737', '23CHLC18504',  # Additional exclusions from screenshot
    '23CHLC22869', '23CHLC26147',
    '23NWLC30820', '23NWLC32904', '24CHLC01046', '24CHLC07523', '24CHLC07816',
    '24CHLC08010', '24CHLC08164', '24CHLC10633', '24CHLC12162', '24CHLC13411',
    '24CHLC15443', '24CHLC15469', '24CHLC15493', '24CHLC16272', '24CHLC17892',
    '24CHLC18130', '24CHLC18229', '24CHLC18921', '24CHLC19088', '24CHLC19890',
    '24CHLC21170', '24CHLC21219', '24CHLC21931', '24NWLC29406', '24NWLC30317',
    '24NWLC30715', '23CHLC04088', '23CHLC12118', '23CHLC12580', '23NWLC33271',
    '24CHLC00593', '24NWLC23018', '24NWLC34626', '23NWLC33801', '23NWLC33883',
    '23NWLC37974', '24CHLC00373', '24NWLC11999', '24NWLC12798', '24NWLC14122',
    '24NWLC14167', '24NWLC14172', '24NWLC15387', '24NWLC17337', '24NWLC17653',
    '24NWLC18762', '24NWLC18973', '24NWLC20168', '24NWLC21288', '24NWLC21396',
    '24NWLC22737', '24NWLC23489', '24NWLC23633', '24NWLC23772', '24NWLC23926',
    '24NWLC24410', '24NWLC24563', '24NWLC26480', '24NWLC27295', '24NWLC28395',
    '24NWLC28521', '24NWLC28537', '23NWLC29036', '23CHLC28557', '23NWLC25562',
    '24CHLC22253', '24CHLC23603', '24CHLC24417', '24CHLC25192', '24CHLC25589',
    '24CHLC27224', '24CHLC28312', '24CHLC28586', '24CHLC28620', '24CHLC29276',
    '24CHLC29300', '24CHLC30395', '24CHLC30453', '24CHLC33731', '24CHLC36483',
    '24NWLC00314', '24NWLC00859', '24NWLC03100', '24NWLC04735', '24NWLC07718',
    '24NWLC08314', '24NWLC08754', '24NWLC09091', '24NWLC10824', '24NWLC11342',
    '24NWLC32406', '24NWLC32801', '24NWLC32884', '24NWLC38337'
]

# Read the CSV file
file_path = '/Users/othmanbensouda/Desktop/debt_collection_website/files/results_gold_rows.csv'
df = pd.read_csv(file_path)

# Display initial info
print("Initial shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nFirst few rows:")
print(df.head())

# Filter out excluded case numbers
print(f"\nFiltering out {len(exclude_cases)} case numbers...")
df = df[~df['case_number'].isin(exclude_cases)]
print(f"Shape after exclusion: {df.shape}")

# Filter to keep only specific annotators
allowed_annotators = ['Parker', 'Brian', 'Victor']
print(f"\nKeeping only annotators: {allowed_annotators}")
df = df[df['annotator_id'].isin(allowed_annotators)]
print(f"Shape after annotator filter: {df.shape}")

# Check if 'version' column exists
if 'version' in df.columns:
    # Group by case_number and round, keep the row with max version
    df_filtered = df.loc[df.groupby(['case_number', 'round'])['version'].idxmax()]
    
    print("\n" + "="*50)
    print("After filtering (keeping max version):")
    print("="*50)
    print("Filtered shape:", df_filtered.shape)
    print("\nFirst few rows of filtered data:")
    print(df_filtered.head(10))
    
    # Save the filtered data
    output_path = '/Users/othmanbensouda/Desktop/debt_collection_website/files/filtered_results_aviv.csv'
    df_filtered.to_csv(output_path, index=False)
    print(f"\nFiltered data saved to: {output_path}")
    
    # Count unique case numbers
    unique_cases = df_filtered['case_number'].nunique()
    print(f"\nNumber of unique case numbers: {unique_cases}")
    
    # Compare with cases_assigned.xlsx
    print("\n" + "="*50)
    print("Comparing with cases_assigned.xlsx")
    print("="*50)
    
    try:
        # Read the assigned cases file
        assigned_path = '/Users/othmanbensouda/Desktop/debt_collection_website/files/cases_assigned.xlsx'
        df_assigned = pd.read_excel(assigned_path)
        
        # Get unique case numbers from both datasets
        filtered_cases = set(df_filtered['case_number'].unique())
        assigned_cases = set(df_assigned['case_number'].unique())
        
        # Find cases in filtered but not in assigned
        cases_not_in_assigned = filtered_cases - assigned_cases
        
        print(f"\nCases in filtered results: {len(filtered_cases)}")
        print(f"Cases in assigned file: {len(assigned_cases)}")
        print(f"Cases in filtered but NOT in assigned: {len(cases_not_in_assigned)}")
        
        if cases_not_in_assigned:
            print("\nList of cases not in assigned file:")
            for case in sorted(cases_not_in_assigned):
                print(f"  - {case}")
            
            # Save these cases to a file
            not_assigned_output = '/Users/othmanbensouda/Desktop/debt_collection_website/files/cases_not_in_assigned.txt'
            with open(not_assigned_output, 'w') as f:
                for case in sorted(cases_not_in_assigned):
                    f.write(f"{case}\n")
            print(f"\nList saved to: {not_assigned_output}")
        else:
            print("\nAll filtered cases are in the assigned file!")
            
    except Exception as e:
        print(f"\nError comparing with assigned cases: {e}")
    
else:
    print("\nWarning: 'version' column not found in the CSV")
    print("Available columns:", df.columns.tolist())

Initial shape: (566, 102)

Column names:
['case_number', 'annotator_id', 'created_at', 'time_caselevel', 'rfdj_demand_amount', 'rfdj_interest_amount', 'rfdj_cost_amount', 'rfdj_attorney_fees_amount', 'complaint_has_prayer', 'rfdjamount_1', 'allegation_debt_buyer_1', 'allegation_nature_of_debt_2', 'allegation_sole_owner_3', 'alleges_chargeoff_balance_4', 'alleged_chargeoff_balance_4', 'alleges_last_payment_date_5', 'alleged_last_payment_date_5', 'alleges_default_date_5b', 'alleged_default_date_5b', 'alleges_chargeoff_creditor_info_6', 'alleged_chargeoff_creditor_info_6', 'alleges_debtor_last_known_address_7', 'alleged_debtor_last_known_address_7', 'alleges_postchargeoffpurchaserinfo_0', 'postchargeoffpurchaserinfo_0', 'allegation_1788_52_compliance', 'hascontractorlaststatement_0', 'complaint_agreement_ref', 'declaration_has_agreement_proof', 'declaration_agreement_ref', 'ownership_chain_sufficient', 'ownership_chain_refs', 'haschargeoffbalance_1', 'chargeoff_balance_ref', 'lastpaymentd